# Download minGPT

In [1]:
!git clone https://github.com/karpathy/minGPT.git

Cloning into 'minGPT'...
remote: Enumerating objects: 489, done.
remote: Counting objects: 100% (239/239), done.
remote: Compressing objects: 100% (94/94), done.
remote: Total 489 (delta 153), reused 145 (delta 145), pack-reused 250 (from 1)
Receiving objects: 100% (489/489), 1.43 MiB | 9.72 MiB/s, done.
Resolving deltas: 100% (267/267), done.


In [2]:
import torch

In [3]:
%cd minGPT/

/content/minGPT


In [ ]:
%pip install -e .

Obtaining file:///content/minGPT
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 56.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvi

# Test demo.ipynb

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from mingpt.utils import set_seed
set_seed(3407)

In [ ]:
import pickle

class SortDataset(Dataset):
    """
    Dataset for the Sort problem. E.g. for problem length 6:
    Input: 0 0 2 1 0 1 -> Output: 0 0 0 1 1 2
    Which will feed into the transformer concatenated as:
    input:  0 0 2 1 0 1 0 0 0 1 1
    output: I I I I I 0 0 0 1 1 2
    where I is "ignore", as the transformer is reading the input sequence
    """

    def __init__(self, split, length=6, num_digits=3):
        assert split in {'train', 'test'}
        self.split = split
        self.length = length
        self.num_digits = num_digits

    def __len__(self):
        return 10000 # ...

    def get_vocab_size(self):
        return self.num_digits

    def get_block_size(self):
        # the length of the sequence that will feed into transformer,
        # containing concatenated input and the output, but -1 because
        # the transformer starts making predictions at the last input element
        return self.length * 2 - 1

    def __getitem__(self, idx):

        # use rejection sampling to generate an input example from the desired split
        while True:
            # generate some random integers
            inp = torch.randint(self.num_digits, size=(self.length,), dtype=torch.long)
            # half of the time let's try to boost the number of examples that
            # have a large number of repeats, as this is what the model seems to struggle
            # with later in training, and they are kind of rate
            if torch.rand(1).item() < 0.5:
                if inp.unique().nelement() > self.length // 2:
                    # too many unqiue digits, re-sample
                    continue
            # figure out if this generated example is train or test based on its hash
            h = hash(pickle.dumps(inp.tolist()))
            inp_split = 'test' if h % 4 == 0 else 'train' # designate 25% of examples as test
            if inp_split == self.split:
                break # ok

        # solve the task: i.e. sort
        sol = torch.sort(inp)[0]

        # concatenate the problem specification and the solution
        cat = torch.cat((inp, sol), dim=0)

        # the inputs to the transformer will be the offset sequence
        x = cat[:-1].clone()
        y = cat[1:].clone()
        # we only want to predict at output locations, mask out the loss at the input locations
        y[:self.length-1] = -1
        return x, y


In [ ]:
# print an example instance of the dataset
train_dataset = SortDataset('train')
test_dataset = SortDataset('test')
x, y = train_dataset[0]
for a, b in zip(x,y):
    print(int(a),int(b))

1 -1
0 -1
1 -1
0 -1
0 -1
0 0
0 0
0 0
0 0
0 1
1 1


In [ ]:
# create a GPT instance
from mingpt.model import GPT

model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()
model = GPT(model_config)

number of parameters: 0.09M


In [ ]:
# create a Trainer object
from mingpt.trainer import Trainer

train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4 # the model we're using is so small that we can go a bit faster
train_config.max_iters = 2000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)

running on device cpu


In [ ]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter_dt {trainer.iter_dt * 1000:.2f}ms; iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

trainer.run()

iter_dt 0.00ms; iter 0: train loss 1.06342
iter_dt 58.58ms; iter 100: train loss 0.14315
iter_dt 38.47ms; iter 200: train loss 0.07915
iter_dt 38.17ms; iter 300: train loss 0.03190
iter_dt 59.46ms; iter 400: train loss 0.04693
iter_dt 39.49ms; iter 500: train loss 0.02466
iter_dt 37.90ms; iter 600: train loss 0.02111
iter_dt 56.39ms; iter 700: train loss 0.05290
iter_dt 37.25ms; iter 800: train loss 0.01741
iter_dt 37.93ms; iter 900: train loss 0.04121
iter_dt 51.73ms; iter 1000: train loss 0.01718
iter_dt 56.43ms; iter 1100: train loss 0.00328
iter_dt 37.44ms; iter 1200: train loss 0.00663
iter_dt 64.51ms; iter 1300: train loss 0.00455
iter_dt 40.49ms; iter 1400: train loss 0.00203
iter_dt 41.01ms; iter 1500: train loss 0.00430
iter_dt 57.39ms; iter 1600: train loss 0.00253
iter_dt 46.43ms; iter 1700: train loss 0.00389
iter_dt 38.45ms; iter 1800: train loss 0.00583
iter_dt 38.34ms; iter 1900: train loss 0.00162


In [ ]:
# now let's perform some evaluation
model.eval();

In [ ]:
def eval_split(trainer, split, max_batches):
    dataset = {'train':train_dataset, 'test':test_dataset}[split]
    n = train_dataset.length # naugy direct access shrug
    results = []
    mistakes_printed_already = 0
    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)
    for b, (x, y) in enumerate(loader):
        x = x.to(trainer.device)
        y = y.to(trainer.device)
        # isolate the input pattern alone
        inp = x[:, :n]
        sol = y[:, -n:]
        # let the model sample the rest of the sequence
        cat = model.generate(inp, n, do_sample=False) # using greedy argmax, not sampling
        sol_candidate = cat[:, n:] # isolate the filled in sequence
        # compare the predicted sequence to the true sequence
        correct = (sol == sol_candidate).all(1).cpu() # Software 1.0 vs. Software 2.0 fight RIGHT on this line haha
        for i in range(x.size(0)):
            results.append(int(correct[i]))
            if not correct[i] and mistakes_printed_already < 3: # only print up to 5 mistakes to get a sense
                mistakes_printed_already += 1
                print("GPT claims that %s sorted is %s but gt is %s" % (inp[i].tolist(), sol_candidate[i].tolist(), sol[i].tolist()))
        if max_batches is not None and b+1 >= max_batches:
            break
    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100*rt.mean()))
    return rt.sum()

# run a lot of examples from both train and test through the model and verify the output correctness
with torch.no_grad():
    train_score = eval_split(trainer, 'train', max_batches=50)
    test_score  = eval_split(trainer, 'test',  max_batches=50)

train final score: 5000/5000 = 100.00% correct
test final score: 5000/5000 = 100.00% correct


In [ ]:
# let's run a random given sequence through the model as well
n = train_dataset.length # naugy direct access shrug
inp = torch.tensor([[0, 0, 2, 1, 0, 1]], dtype=torch.long).to(trainer.device)
assert inp[0].nelement() == n
with torch.no_grad():
    cat = model.generate(inp, n, do_sample=False)
sol = torch.sort(inp[0])[0]
sol_candidate = cat[:, n:]
print('input sequence  :', inp.tolist())
print('predicted sorted:', sol_candidate.tolist())
print('gt sort         :', sol.tolist())
print('matches         :', bool((sol == sol_candidate).all()))

input sequence  : [[0, 0, 2, 1, 0, 1]]
predicted sorted: [[0, 0, 0, 1, 1, 2]]
gt sort         : [0, 0, 0, 1, 1, 2]
matches         : True


# Create probability

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from mingpt.utils import set_seed
from mingpt.model import GPT
from mingpt.trainer import Trainer
import torch.nn.functional as F
set_seed(3407)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)  # Move the model to the correct device


In [ ]:
class BinarySequenceDataset(Dataset):
    def __init__(self, length=8, num_sequences=10000):
        self.length = length
        self.num_sequences = num_sequences
        self.vocab_size = 2  # Binary: 0 and 1

    def __len__(self):
        return self.num_sequences

    def get_vocab_size(self):
        return self.vocab_size

    def get_block_size(self):
        return self.length

    def __getitem__(self, idx):
        # Random binary sequence
        sequence = torch.randint(0, 2, size=(self.length,), dtype=torch.long)

        # The input to the model is the sequence excluding the last bit
        x = sequence[:-1]
        # The target is the sequence excluding the first bit (shifted by 1)
        y = sequence[1:]

        return x, y


# Generate certain sequence of P for sequence


In [ ]:
# Define the GPT model
train_dataset = BinarySequenceDataset(length=8)
model_config = GPT.get_default_config()
model_config.model_type = 'gpt-nano'
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()

# Initialize the GPT model
model = GPT(model_config)


number of parameters: 0.09M


In [ ]:
train_config = Trainer.get_default_config()
train_config.learning_rate = 5e-4
train_config.max_iters = 2000
train_config.num_workers = 0
trainer = Trainer(train_config, model, train_dataset)


running on device cpu


In [ ]:
def batch_end_callback(trainer):
    if trainer.iter_num % 100 == 0:
        print(f"iter {trainer.iter_num}: train loss {trainer.loss.item():.5f}")
trainer.set_callback('on_batch_end', batch_end_callback)

trainer.run()


iter 0: train loss 0.69322
iter 100: train loss 0.69295
iter 200: train loss 0.69348
iter 300: train loss 0.69360
iter 400: train loss 0.69237
iter 500: train loss 0.69289
iter 600: train loss 0.69289
iter 700: train loss 0.69328
iter 800: train loss 0.69214
iter 900: train loss 0.69349
iter 1000: train loss 0.69296
iter 1100: train loss 0.69337
iter 1200: train loss 0.69322
iter 1300: train loss 0.69324
iter 1400: train loss 0.69286
iter 1500: train loss 0.69305
iter 1600: train loss 0.69316
iter 1700: train loss 0.69300
iter 1800: train loss 0.69342
iter 1900: train loss 0.69188


In [ ]:
def compute_probability(model, sequence, device):
    model.eval()

    # Move the input sequence to the correct device
    sequence = torch.tensor(sequence, dtype=torch.long).unsqueeze(0).to(device)

    with torch.no_grad():
        # Generate the model's prediction for the next bit in the sequence
        output = model(sequence)

        # The output is a tuple, so get the logits (first element of the tuple)
        logits = output[0]  # Shape: (batch_size, seq_len, vocab_size)

        # Apply softmax to get probabilities (logits -> probs)
        probs = torch.nn.functional.softmax(logits, dim=-1)  # Apply softmax on vocab dimension

        # The last token's prediction will be at the last index (seq_len - 1)
        next_bit_probs = probs[0, -1]  # Probabilities for the next token (last token in sequence)

        # Extract the probability of the next bit being 1
        next_bit_prob = next_bit_probs[1].item()  # Index 1 corresponds to the probability of 1

        return next_bit_prob


In [ ]:
test_sequence = [1, 1, 1, 1, 1, 1, 1, 1]  # Example binary sequence
probability = compute_probability(model, test_sequence, device)
print(f"Probability of next bit in sequence {test_sequence} being 1: {probability:.4f}")


Probability of next bit in sequence [1, 1, 1, 1, 1, 1, 1, 1] being 1: 0.4989


# Create


In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class BinarySequenceDataset(Dataset):
    def __init__(self, sequences, block_size):
        self.sequences = sequences
        self.block_size = block_size

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        sequence = self.sequences[idx]
        # Ensure the sequence is at least block_size long for context
        if len(sequence) < self.block_size + 1:
            # Pad or handle short sequences as needed
            padding = [0] * (self.block_size + 1 - len(sequence))
            sequence = sequence + padding
        # Take a chunk of block_size as input and the next bit as target
        x = torch.tensor(sequence[:self.block_size], dtype=torch.long)
        y = torch.tensor(sequence[1:self.block_size + 1], dtype=torch.long)
        return x, y

def generate_synthetic_binary_data(num_sequences=1000, seq_length=20):
    return [[np.random.randint(0, 2) for _ in range(seq_length)] for _ in range(num_sequences)]

# Example usage:
block_size = 16  # Context length for the transformer
raw_sequences = generate_synthetic_binary_data(num_sequences=1000)
train_dataset = BinarySequenceDataset(raw_sequences, block_size)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
%cd ..

/content/minGPT
